# Neural Melody Generator — Progress Log

## Files i'm uploading on github

.gitignore — the updated exclusion rules ✅

01_setup_check.ipynb, 02_data_exploration.ipynb — your working notebooks ✅

generated_melody_v1.mid — your first AI-generated melody ✅

## Step 1 & 2: Setup Phase


1. **Created project folder and conda environment** — set up a local conda environment (`./env`) inside the project directory with Python and core packages (numpy, matplotlib, jupyter).
2. **Installed PyTorch** — hit a conda archive-extraction bug (Windows-specific, likely antivirus interference); clearing the package cache didn't help, so switched to installing PyTorch via `pip` from PyTorch's official CPU wheel index instead, which worked cleanly.
3. **Installed remaining ML/music libraries** — `music21`, `pretty_midi`, and `jupyter` for MIDI parsing and notebook work.
4. **Verified the environment** — ran a test notebook (`01_setup_check.ipynb`) importing torch, music21, pretty_midi, and numpy to confirm everything works together.
5. **Installed Git** (wasn't on the system yet) and initialized version control — `git init` in the project folder, added a `.gitignore` to exclude the environment folder and MIDI data files from version control.
6. **Set up GitHub remote and pushed first commit** — created the `Neural_Melody_Generator` repository on GitHub, connected it via HTTPS auth, and pushed the initial commit containing the environment setup and verification notebook

## Step 3: Data Pipeline Phase



7. **Loaded and inspected real MIDI data** — confirmed Lakh MIDI files contain multiple instrument tracks per song (9 in the test case), with unreliable instrument name labels (e.g., drum tracks mislabeled "Acoustic Grand Piano").
8. **Selected melodic track** — chose the non-drum track with the most notes as the primary melody line for the first model version (decided to simplify to melody-only rather than modeling full chords, as a V1 approach).
9. **Built chord-to-melody reduction** — collapsed simultaneous notes (chords) into a single note (highest pitch). Iterated on the time-tolerance window (50ms → 150ms) after testing showed notes belonging to the same chord were spread further apart than expected in real performance data.
10. **Built reusable `extract_melody_from_midi()` function** — wraps chord-to-melody extraction with error handling for corrupted/malformed files.
11. **Tested pipeline on a sample of 50 songs** — 49/50 processed successfully (98% success rate), confirming the pipeline handles real-world messy data gracefully.
12. **Built vocabulary** — mapped 65 unique MIDI pitches (range 25–93) to integer tokens, similar to word→ID mapping in NLP.
13. **Tokenized all processed songs** — converted each song's melody into a sequence of integer tokens.
14. **Created training sequences** — used a sliding window (50 notes → predict next note) to generate input/target pairs, producing 26,377 training examples from 49 songs.

# Step 4: Model Training Phase

15. **Diagnosed overfitting in the first training run** — the initial run had no validation set, and the loss was suspiciously low (~0.08). Generating from a training seed reproduced the original sequence note-for-note, which showed the model was memorizing rather than generalizing. The cause: a sliding window with a stride of 1 turns 49 songs into 26,377 examples that overlap by 49 of 50 notes, so the model sees near-duplicate data many times.
16. **Added a train/validation split (80/20) and retrained from a fresh model** — this gave a more realistic loss curve (train: 1.77→0.27, val: 1.24→0.30) with train and validation loss falling together. *Limitation:* the split was random across windows, so overlapping windows from the same song can land on both sides. The validation loss is therefore likely optimistic. Planned fix: split by song, so no song appears in both sets.
17. Generated melodies from the trained model — confirmed the model produces genuine melodic variation (not verbatim memorized output) when given a musically varied seed. Compared temperature 1.0 vs 1.2, observing more conservative/stable output at lower temperature and occasional "risky" pitch jumps at higher temperature — demonstrating the exploration/coherence trade-off in sampling.
18. Converted generated tokens into a playable `.mid` file — built a `tokens_to_midi()` function using `pretty_midi` to reconstruct actual MIDI notes from the model's output tokens, producing `generated_melody_v1.mid` as the first tangible, listenable output of the project.

# Step 5: Deployment Phase

19. **Saved the trained model and vocabulary** — exported the LSTM weights (`melody_lstm.pth`) and the pitch↔token mappings (`vocab.pkl`) so the app can generate melodies without retraining.
20. **Built a Streamlit demo app (`app.py`)** — loads the saved model, lets the user control temperature (creativity) and melody length, generates a melody from a random seed, and offers the result as a downloadable `.mid` file. Used `@st.cache_resource` so the model loads once instead of on every click.
21. **Tested the app locally** — confirmed generation and download work end-to-end at `localhost:8501`.
22. **Deployed to Hugging Face Spaces** — pending

# Progress

- ✅ **A short AI-generated tune**: done (`generated_melody_v1.mid`)
- 🟡 **A live demo link**: Streamlit app built and tested locally; deploying to Hugging Face Spaces is next
- 🟡 **A polished GitHub repo**: solid commit history, data pipeline, trained model and app code in place; README, loss-curve chart and sample outputs still pending

(C:\Users\chSw907\Documents\Neural_Melody_Generator\env) C:\Users\chSw907\Documents\Neural_Melody_Generator>streamlit run app.py

## Next steps

Finish deploying the LSTM demo to Hugging Face Spaces 
Write the first README draft and push it, so the repo is presentable
Build the Transformer, which is the next big phase